# 08 - Gemma QLoRA Fine-Tuning

Bu notebook Gemma-4-E2B-it için **Causal LM + QLoRA** denemesi yapar.

Önemli: Gemma 4, `AutoModelForSequenceClassification` desteklemediği için Qwen'deki sequence classification yöntemi burada kullanılmaz. Modelden çıktı olarak doğrudan `0` veya `1` üretmesi öğretilir.

Bu notebook'ta `prepare_model_for_kbit_training(model)` özellikle kullanılmaz. Çünkü T4 GPU üzerinde OOM verdi.


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime ayarlarından T4 GPU açılmalı.")

CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -U git+https://github.com/huggingface/transformers.git -q
!pip install datasets accelerate peft bitsandbytes scikit-learn pandas torchao -q
!pip install -U torchao -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.1 MB/s eta 0:00:00


In [3]:
import os
import gc
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoProcessor,
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("Imports ready.")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Imports ready.
CUDA available: True
GPU: Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
drive_project_dir = "/content/drive/MyDrive/turkish_answer_quality_slm"
drive_tables_dir = "/content/drive/MyDrive/turkish_answer_quality_slm/outputs/tables"
drive_model_dir_pilot = "/content/drive/MyDrive/turkish_answer_quality_slm/gemma_strategy_b_qlora_pilot"
drive_model_dir_longrun = "/content/drive/MyDrive/turkish_answer_quality_slm/gemma_strategy_b_qlora_longrun"

os.makedirs(drive_project_dir, exist_ok=True)
os.makedirs(drive_tables_dir, exist_ok=True)
os.makedirs(drive_model_dir_pilot, exist_ok=True)
os.makedirs(drive_model_dir_longrun, exist_ok=True)

print("Drive project dir:", drive_project_dir)
print("Tables dir:", drive_tables_dir)
print("Pilot checkpoint dir:", drive_model_dir_pilot)
print("Long-run checkpoint dir:", drive_model_dir_longrun)

Drive project dir: /content/drive/MyDrive/turkish_answer_quality_slm
Tables dir: /content/drive/MyDrive/turkish_answer_quality_slm/outputs/tables
Pilot checkpoint dir: /content/drive/MyDrive/turkish_answer_quality_slm/gemma_strategy_b_qlora_pilot
Long-run checkpoint dir: /content/drive/MyDrive/turkish_answer_quality_slm/gemma_strategy_b_qlora_longrun


## Veri dosyalarını yükleme

In [6]:
train_path = "/data/processed/strategy_b/train.csv"
validation_path = "/data/processed/strategy_b/validation.csv"
test_path = "/data/processed/strategy_b/test.csv"

print("Train exists:", os.path.exists(train_path))
print("Validation exists:", os.path.exists(validation_path))
print("Test exists:", os.path.exists(test_path))

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())
print("\nValidation label distribution:")
print(validation_df["label"].value_counts().sort_index())
print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

Train exists: True
Validation exists: True
Test exists: True
Train shape: (11575, 8)
Validation shape: (1447, 8)
Test shape: (1447, 8)

Train label distribution:
label
0    7085
1    4490
Name: count, dtype: int64

Validation label distribution:
label
0    886
1    561
Name: count, dtype: int64

Test label distribution:
label
0    885
1    562
Name: count, dtype: int64


In [7]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "label"]])
validation_dataset = Dataset.from_pandas(validation_df[["input_text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "label"]])

print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['input_text', 'label'],
    num_rows: 11575
})
Dataset({
    features: ['input_text', 'label'],
    num_rows: 1447
})
Dataset({
    features: ['input_text', 'label'],
    num_rows: 1447
})


In [8]:
model_id = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(model_id)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
except Exception:
    tokenizer = processor.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Processor loaded:", model_id)
print("Tokenizer pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Processor loaded: google/gemma-4-E2B-it
Tokenizer pad token: <pad>
Pad token id: 0


## Causal LM eğitim formatı

Gemma sequence classification desteklemediği için eğitim formatı şöyle olacak:

Prompt:

`Classify the Turkish student answer... Input: ... Label:`

Target:

`0` veya `1`

Loss sadece target label tokenları üzerinde hesaplanır. Prompt tokenları `-100` ile maskelenir.


In [10]:
MAX_LENGTH = 512

SYSTEM_PROMPT = """Classify the Turkish student answer quality.

0 = not high-quality
1 = high-quality

Return only one number: 0 or 1.

Input:
"""

def build_prompt(input_text):
    return SYSTEM_PROMPT + str(input_text).strip() + "\n\nLabel:"


def tokenize_for_causal_lm(example):
    prompt = build_prompt(example["input_text"])
    target = " " + str(int(example["label"])) + tokenizer.eos_token

    target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]
    max_prompt_len = MAX_LENGTH - len(target_ids)

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=max_prompt_len
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + target_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

tokenized_train = train_dataset.map(
    tokenize_for_causal_lm,
    remove_columns=train_dataset.column_names
)

tokenized_validation = validation_dataset.map(
    tokenize_for_causal_lm,
    remove_columns=validation_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_for_causal_lm,
    remove_columns=test_dataset.column_names
)

print(tokenized_train)
print(tokenized_validation)
print(tokenized_test)

Map:   0%|          | 0/11575 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 11575
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1447
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1447
})


In [11]:
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print("4-bit Gemma causal LM loaded.")
print("Pad token id:", model.config.pad_token_id)

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

4-bit Gemma causal LM loaded.
Pad token id: 0


Aşağıdaki bölümde `prepare_model_for_kbit_training` kullanılmıyor. Çünkü önceki denemede T4 GPU üzerinde OOM verdi.

In [12]:
model.gradient_checkpointing_enable()

try:
    model.enable_input_require_grads()
    print("Input gradients enabled.")
except Exception as e:
    print("Could not enable input gradients:", e)

print("Gradient checkpointing enabled.")

Input gradients enabled.
Gradient checkpointing enabled.


In [13]:
for name, module in model.named_modules():
    if "proj" in name or "linear" in name:
        print(name)

model.vision_tower.patch_embedder.input_proj
model.vision_tower.encoder.layers.0.self_attn.q_proj
model.vision_tower.encoder.layers.0.self_attn.q_proj.linear
model.vision_tower.encoder.layers.0.self_attn.k_proj
model.vision_tower.encoder.layers.0.self_attn.k_proj.linear
model.vision_tower.encoder.layers.0.self_attn.v_proj
model.vision_tower.encoder.layers.0.self_attn.v_proj.linear
model.vision_tower.encoder.layers.0.self_attn.o_proj
model.vision_tower.encoder.layers.0.self_attn.o_proj.linear
model.vision_tower.encoder.layers.0.mlp.gate_proj
model.vision_tower.encoder.layers.0.mlp.gate_proj.linear
model.vision_tower.encoder.layers.0.mlp.up_proj
model.vision_tower.encoder.layers.0.mlp.up_proj.linear
model.vision_tower.encoder.layers.0.mlp.down_proj
model.vision_tower.encoder.layers.0.mlp.down_proj.linear
model.vision_tower.encoder.layers.1.self_attn.q_proj
model.vision_tower.encoder.layers.1.self_attn.q_proj.linear
model.vision_tower.encoder.layers.1.self_attn.k_proj
model.vision_tower.e

In [14]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

print("QLoRA applied.")
model.print_trainable_parameters()

ValueError: Target module Gemma4ClippableLinear(
  (linear): Linear4bit(in_features=768, out_features=768, bias=False)
) is not supported. Currently, only the following modules are supported: `torch.nn.Linear`, `torch.nn.Embedding`, `torch.nn.Conv1d`, `torch.nn.Conv2d`, `torch.nn.Conv3d`, `transformers.pytorch_utils.Conv1D`, `torch.nn.MultiheadAttention.`.

## Gemma Fine-Tuning Status

Gemma-4-E2B-it was tested for fine-tuning after the zero-shot evaluation stage.

Two fine-tuning approaches were attempted:

1. Sequence classification fine-tuning with AutoModelForSequenceClassification
2. Causal LM QLoRA fine-tuning with AutoModelForCausalLM

The sequence classification approach failed because Gemma4Config is not currently supported by AutoModelForSequenceClassification.

The causal LM QLoRA approach also failed because Gemma 4 uses Gemma4ClippableLinear modules, which are not supported as LoRA target modules by the current PEFT setup.

Therefore, Gemma fine-tuning was not completed under the available Colab T4 environment. Gemma will be kept in the benchmark as a zero-shot evaluated model, while Qwen will represent the successfully fine-tuned larger model.

# ***Gemma-4-E2B-it fine-tuning was attempted but could not be completed under the available Colab T4 environment due to current model architecture and PEFT compatibility limitations.***

In [15]:
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

print("Data collator ready.")

Data collator ready.


## Pilot training

Önce kısa pilot çalıştırılır. Amaç final skor almak değil, Gemma Causal LM + QLoRA pipeline'ının çalışıp çalışmadığını görmektir.


In [ ]:
pilot_validation_dataset = tokenized_validation.select(range(100))

pilot_training_args = TrainingArguments(
    output_dir=drive_model_dir_pilot,
    max_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_strategy="steps",
    logging_steps=10,
    fp16=False,
    bf16=False,
    report_to="none",
    save_total_limit=2
)

pilot_trainer = Trainer(
    model=model,
    args=pilot_training_args,
    train_dataset=tokenized_train,
    eval_dataset=pilot_validation_dataset,
    data_collator=data_collator
)

print("Pilot trainer ready.")

In [ ]:
pilot_train_result = pilot_trainer.train()
print("Pilot training completed.")

## Generation-based evaluation helper

Causal LM olduğu için sınıflandırma metriğini `Trainer.evaluate()` doğrudan vermez. Modelin `0` veya `1` üretmesine bakarak ayrıca hesaplanır.


In [ ]:
def parse_prediction(text):
    text = str(text).strip()
    if text.startswith("0"):
        return 0
    if text.startswith("1"):
        return 1
    return None


def generate_prediction_for_input(input_text, max_prompt_tokens=500):
    prompt = build_prompt(input_text)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_prompt_tokens
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()


def evaluate_generation(df, sample_name="sample"):
    rows = []
    for idx, row in df.reset_index(drop=True).iterrows():
        raw_response = generate_prediction_for_input(row["input_text"])
        pred = parse_prediction(raw_response)
        rows.append({
            "label": int(row["label"]),
            "prediction": pred,
            "raw_response": raw_response,
            "Score": row.get("Score", None),
            "soru": row.get("soru", None)
        })
        if (idx + 1) % 10 == 0:
            print(f"Processed {idx + 1}/{len(df)}")

    pred_df = pd.DataFrame(rows)
    valid_df = pred_df.dropna(subset=["prediction"]).copy()
    valid_df["prediction"] = valid_df["prediction"].astype(int)

    results = {
        "sample_name": sample_name,
        "total_examples": len(pred_df),
        "valid_predictions": len(valid_df),
        "invalid_predictions": len(pred_df) - len(valid_df),
        "accuracy": accuracy_score(valid_df["label"], valid_df["prediction"]) if len(valid_df) else None,
        "macro_f1": f1_score(valid_df["label"], valid_df["prediction"], average="macro", zero_division=0) if len(valid_df) else None,
        "weighted_f1": f1_score(valid_df["label"], valid_df["prediction"], average="weighted", zero_division=0) if len(valid_df) else None,
    }

    return results, pred_df

In [ ]:
sample_100_label_0 = test_df[test_df["label"] == 0].sample(n=50, random_state=42)
sample_100_label_1 = test_df[test_df["label"] == 1].sample(n=50, random_state=42)

gemma_eval_100 = pd.concat([sample_100_label_0, sample_100_label_1])
gemma_eval_100 = gemma_eval_100.sample(frac=1, random_state=42).reset_index(drop=True)

pilot_eval_results, pilot_prediction_df = evaluate_generation(gemma_eval_100, sample_name="balanced_100_after_pilot")

print(pilot_eval_results)
print(pilot_prediction_df["prediction"].value_counts(dropna=False))

In [ ]:
pilot_results_path = os.path.join(drive_tables_dir, "gemma_qlora_pilot_generation_results.csv")
pilot_predictions_path = os.path.join(drive_tables_dir, "gemma_qlora_pilot_generation_predictions.csv")

pd.DataFrame([pilot_eval_results]).to_csv(pilot_results_path, index=False)
pilot_prediction_df.to_csv(pilot_predictions_path, index=False)

print("Saved:")
print(pilot_results_path)
print(pilot_predictions_path)

## Long-run training

Pilot çalışırsa bu bölüm kullanılacak. Pilot başarısızsa veya OOM verirse long-run başlatma.


In [ ]:
long_validation_dataset = tokenized_validation.select(range(300))

long_training_args = TrainingArguments(
    output_dir=drive_model_dir_longrun,
    max_steps=600,
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    logging_strategy="steps",
    logging_steps=20,
    fp16=False,
    bf16=False,
    report_to="none",
    save_total_limit=3
)

long_trainer = Trainer(
    model=model,
    args=long_training_args,
    train_dataset=tokenized_train,
    eval_dataset=long_validation_dataset,
    data_collator=data_collator
)

print("Long-run trainer ready.")

In [ ]:
long_train_result = long_trainer.train()
print("Long-run training completed.")

In [ ]:
long_eval_results, long_prediction_df = evaluate_generation(gemma_eval_100, sample_name="balanced_100_after_longrun")

print(long_eval_results)
print(long_prediction_df["prediction"].value_counts(dropna=False))

In [ ]:
long_results_path = os.path.join(drive_tables_dir, "gemma_qlora_longrun_generation_results.csv")
long_predictions_path = os.path.join(drive_tables_dir, "gemma_qlora_longrun_generation_predictions.csv")

pd.DataFrame([long_eval_results]).to_csv(long_results_path, index=False)
long_prediction_df.to_csv(long_predictions_path, index=False)

print("Saved:")
print(long_results_path)
print(long_predictions_path)